# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Dataset title: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")
print(f"Identifier: {getattr(metadata, 'identifier', 'N/A')}")


## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List available record sets and their fields using their @id.

# Get all record set objects
record_sets = list(dataset.record_sets)

print("Available Record Sets (@id, name):")
for rs in record_sets:
    print(f"@id: {rs.id}, name: {rs.name}")

# For demonstration, show all fields in the first record set
if record_sets:
    recset = record_sets[0]
    print(f"\nFields in record set '{recset.name}' (@id, name, data_type):")
    for field in recset.fields:
        print(f"  @id: {field.id}, name: {field.name}, data_type: {field.data_type}")
else:
    print("No record sets found.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set by @id

# Get all record set @id
record_set_ids = [rs.id for rs in dataset.record_sets]

# For reproducibility, show record set ids
print("Record set @ids:")
print(record_set_ids)

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded DataFrame shape for {record_set_id}: {df.shape}")

# For demonstration, select the main record set used for analysis
main_record_set_id = record_set_ids[0]  # Change this to the main record set if necessary
print(f"\nColumns in '{main_record_set_id}':")
print(dataframes[main_record_set_id].columns.tolist())

# Show the first few records
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Example EDA on a numeric field (e.g., GAD-7 total score)
# Replace field IDs below with the ones listed in the previous overview

# Get all field @ids of the main record set
main_record_set = None
for rs in dataset.record_sets:
    if rs.id == main_record_set_id:
        main_record_set = rs

if main_record_set is None:
    raise ValueError("Main record set not found.")

# Find a numeric field id, e.g. total score for GAD-7 or PHQ-9
numeric_field_id = None
for field in main_record_set.fields:
    if field.data_type in ('Float', 'Number', 'Integer') and ('gad' in field.name.lower() or 'phq' in field.name.lower()):
        numeric_field_id = field.id
        break

if numeric_field_id is None:
    # Just take any Integer/Float field as fallback
    for field in main_record_set.fields:
        if field.data_type in ('Float', 'Number', 'Integer'):
            numeric_field_id = field.id
            break

print(f"Selected numeric field for analysis: {numeric_field_id}")

df = dataframes[main_record_set_id]

# Filter records where the score is above a threshold
threshold = 10
if numeric_field_id in df.columns:
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}: {len(filtered_df)} records")
    display(filtered_df.head())

    # Normalizing the field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Filtered and normalized {numeric_field_id}:\n", filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Find a group field (e.g., gender/sex or level of education)
    group_field_id = None
    for field in main_record_set.fields:
        if (('gender' in field.name.lower() or 'sex' in field.name.lower() or 'education' in field.name.lower()) and field.data_type == 'Text'):
            group_field_id = field.id
            break

    if group_field_id is not None and group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index().sort_values(numeric_field_id, ascending=False)
        print(f"\nMean {numeric_field_id} by {group_field_id}:")
        print(grouped_df)
    else:
        print("No suitable group field found for grouping.")
else:
    print(f"Field {numeric_field_id} not found in DataFrame columns.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the distribution of the numeric field
if numeric_field_id in df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.show()

    # If grouping field is found, show boxplots
    if group_field_id is not None and group_field_id in df.columns:
        plt.figure(figsize=(10, 6))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

**Key Findings:**
- This FAIR² dataset provides mental health indicators and demographic information from Kilifi County, Kenya, following a Croissant schema for enhanced interoperability.
- The main survey record set encapsulates fields such as GAD-7, PHQ-9, and PSQ scores, ideal for quantitative analysis of psychological health.
- Data filtering and normalization enable robust analysis—e.g., for high GAD-7/PHQ-9 scores by demographic group.
- Visualizations show value distributions and differences across groups, highlighting areas for potential public health interventions or deeper analyses.

Further work may include handling missing values, advanced modeling (e.g., prediction), and correlational analyses based on the clear, modular Croissant schema.